# Rice leaf disease classification with a CNN

Loads an image folder (one sub-folder per class) with `image_dataset_from_directory` and trains a small CNN. The dataset is not included; see the README.


In [ ]:
import tensorflow as tf
# Allocate GPU memory only as needed. By default TensorFlow grabs all of it, which makes a second
# notebook (or kernel) fail with an out-of-memory error on the 4 GB laptop GPU.
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)


In [ ]:
import os
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

DATA_DIR = os.environ.get('RICE_DATA_DIR', 'rice_leaf_diseases')
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Dataset folder '{DATA_DIR}' not found. Put the images in one sub-folder per class "
        "or point RICE_DATA_DIR at them (see README).")

train_ds, test_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, image_size=(128, 128),
    validation_split=0.2, subset='both', seed=42)
classes = train_ds.class_names

x, y = next(iter(train_ds))
plt.imshow(x[0] / 255); plt.title(classes[y[0]]); plt.show()

model = keras.Sequential([
    layers.Rescaling(1./255, input_shape=(128, 128, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(len(classes), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h = model.fit(train_ds, validation_data=test_ds, epochs=20)
print(model.evaluate(test_ds))

plt.plot(h.history['accuracy']); plt.plot(h.history['val_accuracy']); plt.show()
plt.plot(h.history['loss']); plt.plot(h.history['val_loss']); plt.show()